In [ ]:
!pip install -qU \
    langchain-pinecone==0.1.1 \
    langchain-openai==0.1.8 \
    langchain-text-splitters==0.2.0 \
    langchain==0.2.1 \
    pinecone-notebooks==0.1.1


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.3.20 requires langchain<1.0.0,>=0.3.21, but you have langchain 0.2.1 which is incompatible.
langchain-community 0.3.20 requires langchain-core<1.0.0,>=0.3.45, but you have langchain-core 0.2.43 which is incompatible.


In [ ]:
import os
OPENAI_API_KEY = "XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"
os.environ["OPENAI_API_KEY"] =  OPENAI_API_KEY
os.environ["PINECONE_API_KEY"] = "***************************************"
#964e7bbd-2ef0-489b-b08d-e846d2288fc0
#pcsk_3yHy5E_NWAxnDvHsQATx4jNexos78vUiXcprLcTTbMGDMbE2oZ3zMBMwLSNuU4hfNRFpjL


In [ ]:
%pip install -qU langchain_community pypdf

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-openai 0.1.8 requires langchain-core<0.3,>=0.2.2, but you have langchain-core 0.3.45 which is incompatible.
langchain-pinecone 0.1.1 requires langchain-core<0.3,>=0.1.52, but you have langchain-core 0.3.45 which is incompatible.


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

file_path = (
    "********************" #Path to your file
)

loader = PyPDFLoader(file_path)
pages = loader.load_and_split()

In [ ]:
!pip install --upgrade langchain
from langchain.embeddings import OpenAIEmbeddings

model_name = 'text-embedding-ada-002'
#model_name = 'text-embedding-3-large'


embeddings = OpenAIEmbeddings(
    model=model_name,
    openai_api_key=os.environ.get('OPENAI_API_KEY')
)

In [ ]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))

In [ ]:
cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'

spec = ServerlessSpec(cloud=cloud, region=region)
index_name = "multilingual"
!export OPENAI_API_KEY='************************************************'

In [ ]:
import time
import pinecone

# Check if an index with the same name already exists
if index_name in pc.list_indexes().names():
    # If it exists, delete it
    pc.delete_index(index_name)

# Create the index with the correct dimension
pc.create_index(
    name=index_name,
    dimension=1536, # Make sure this matches your embedding dimension
    metric="cosine",
    spec=spec
)

# wait for index to be ready
while not pc.describe_index(index_name).status['ready']:
    time.sleep(1)

from langchain_pinecone import PineconeVectorStore

namespace = "ns0711"

docsearch = PineconeVectorStore.from_documents(
#    documents=md_header_splits,
    documents=pages, # 07/11
    index_name=index_name,
    embedding=embeddings,
    namespace=namespace
)

In [ ]:
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI # import the ChatOpenAI class

llm = ChatOpenAI(
    openai_api_key=os.environ.get('OPENAI_API_KEY'),
  #  model_name='gpt-3.5-turbo',
   model_name='gpt-4o-mini',
    temperature=0.05
)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=docsearch.as_retriever()
)


In [ ]:
# Replace the current code in cell #2
import os
OPENAI_API_KEY = "**************************************************************"  # Replace with your actual API key
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["PINECONE_API_KEY"] = "**********************************************"  # Replace with your actual Pinecone API key

In [ ]:


from google.colab import output, files
from IPython.display import HTML, display, clear_output
import json
import time
import base64

# Create a professional legal document analyzer interface with upload functionality
frontend_html = """
<div style="font-family: 'Inter', sans-serif; max-width: 1200px; margin: 0 auto; padding: 30px; background: #f8fafc; border-radius: 12px; box-shadow: 0 10px 25px rgba(0,0,0,0.05);">
  <h1 style="color: #1e293b; text-align: center; margin-bottom: 25px; font-size: 28px; font-weight: 600;"> LEGAL DOCUMENTS ANALYZER</h1>

  <div style="display: flex; gap: 24px; flex-wrap: wrap;">
    <!-- Left Column: Document Input -->
    <div style="flex: 2; min-width: 320px; background: white; padding: 24px; border-radius: 10px; box-shadow: 0 4px 12px rgba(0,0,0,0.03); border: 1px solid #e2e8f0;">
      <h3 style="color: #334155; margin-top: 0; font-size: 18px; font-weight: 600;">Upload Document</h3>

      <div style="margin: 16px 0; padding: 20px; border: 2px dashed #cbd5e1; border-radius: 8px; text-align: center;" id="drop_zone">
        <input type="file" id="fileInput" style="display: none;" accept=".txt,.pdf,.docx,.doc">
        <label for="fileInput" style="display: block; cursor: pointer;">
          <div style="margin-bottom: 10px;">
            <svg xmlns="http://www.w3.org/2000/svg" width="40" height="40" viewBox="0 0 24 24" fill="none" stroke="#64748b" stroke-width="2" stroke-linecap="round" stroke-linejoin="round">
              <path d="M21 15v4a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2v-4"></path>
              <polyline points="17 8 12 3 7 8"></polyline>
              <line x1="12" y1="3" x2="12" y2="15"></line>
            </svg>
          </div>
          <p style="color: #64748b; margin: 0;">Drag & drop files here or <span style="color: #3b82f6; font-weight: 500;">browse</span></p>
          <p style="color: #94a3b8; font-size: 12px; margin-top: 8px;">Supports TXT files</p>
        </label>
      </div>

      <div style="margin: 20px 0;">
        <div style="display: flex; align-items: center; justify-content: space-between; margin-bottom: 8px;">
          <label for="documentText" style="font-weight: 500; color: #334155; font-size: 14px;">Document Text</label>
          <span id="file_name" style="font-size: 13px; color: #64748b;"></span>
        </div>
        <textarea id="documentText" style="width: 100%; height: 350px; padding: 16px; border: 1px solid #e2e8f0; border-radius: 6px; font-size: 14px; font-family: 'Inter', sans-serif; resize: vertical; line-height: 1.5;"
          placeholder="Paste your legal document content here or upload a file..."></textarea>
      </div>

      <div style="margin-top: 16px;">
        <label for="document_lang" style="display: block; margin-bottom: 8px; font-weight: 500; color: #334155; font-size: 14px;">Document Language</label>
        <select id="document_lang" style="width: 100%; padding: 10px 12px; border: 1px solid #e2e8f0; border-radius: 6px; font-size: 14px; color: #1e293b; background-color: white;">
          <option value="auto">Auto-detect</option>
          <option value="english">English</option>
          <option value="hindi">Hindi</option>
          <option value="tamil">Tamil</option>
          <option value="spanish">Spanish</option>
          <option value="french">French</option>
          <option value="german">German</option>
          <option value="japanese">Japanese</option>
          <option value="chinese">Chinese</option>
          <option value="arabic">Arabic</option>
        </select>
      </div>
    </div>

    <!-- Right Column: Query and Response -->
    <div style="flex: 1; min-width: 280px; background: white; padding: 24px; border-radius: 10px; box-shadow: 0 4px 12px rgba(0,0,0,0.03); border: 1px solid #e2e8f0;">
      <h3 style="color: #334155; margin-top: 0; font-size: 18px; font-weight: 600;">Analysis Options</h3>

      <div style="margin: 16px 0;">
        <label for="analysis_type" style="display: block; margin-bottom: 8px; font-weight: 500; color: #334155; font-size: 14px;">Analysis Type</label>
        <select id="analysis_type" style="width: 100%; padding: 10px 12px; border: 1px solid #e2e8f0; border-radius: 6px; font-size: 14px; color: #1e293b; background-color: white;">
          <option value="summary">Document Summary</option>
          <option value="legal_terms">Identify Legal Terms</option>
          <option value="obligations">Identify Obligations</option>
          <option value="dates">Extract Important Dates</option>
          <option value="parties">Identify Parties</option>
          <option value="custom">Custom Query</option>
        </select>
      </div>

      <div id="custom_query_div" style="margin: 16px 0; display: none;">
        <label for="custom_query" style="display: block; margin-bottom: 8px; font-weight: 500; color: #334155; font-size: 14px;">Custom Query</label>
        <input type="text" id="custom_query" style="width: 100%; padding: 10px 12px; border: 1px solid #e2e8f0; border-radius: 6px; font-size: 14px;"
          placeholder="Ask a specific question...">
      </div>

      <div style="margin: 16px 0;">
        <label for="response_lang" style="display: block; margin-bottom: 8px; font-weight: 500; color: #334155; font-size: 14px;">Response Language</label>
        <select id="response_lang" style="width: 100%; padding: 10px 12px; border: 1px solid #e2e8f0; border-radius: 6px; font-size: 14px; color: #1e293b; background-color: white;">
          <option value="english">English</option>
          <option value="hindi">Hindi</option>
          <option value="tamil">Tamil</option>
          <option value="spanish">Spanish</option>
          <option value="french">French</option>
          <option value="german">German</option>
          <option value="chinese">Chinese</option>
        </select>
      </div>

      <button onclick="analyzeDocument()" style="background: #2563eb; color: white; border: none; padding: 12px 20px;
          width: 100%; border-radius: 6px; font-size: 16px; cursor: pointer; transition: background 0.2s; margin-top: 20px; font-weight: 500;">
        Analyze Document
      </button>
    </div>
  </div>

  <!-- Loading Indicator -->
  <div id="loading" style="display: none; text-align: center; margin-top: 30px;">
    <div style="display: inline-block; width: 40px; height: 40px; border: 3px solid #f3f3f3;
              border-top: 3px solid #3b82f6; border-radius: 50%; animation: spin 1s linear infinite;"></div>
    <p style="color: #64748b; margin-top: 12px; font-size: 15px;">Analyzing document...</p>
  </div>

  <!-- Results Section -->
  <div id="results_container" style="margin-top: 32px; padding: 28px; background: white; border-radius: 10px; box-shadow: 0 4px 12px rgba(0,0,0,0.03); border: 1px solid #e2e8f0; display: none;">
    <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 16px; border-bottom: 1px solid #e2e8f0; padding-bottom: 14px;">
      <h2 style="color: #1e293b; margin: 0; font-size: 20px; font-weight: 600;">Analysis Results</h2>
      <button onclick="downloadResults()" style="background: #f8fafc; color: #334155; border: 1px solid #e2e8f0; padding: 8px 14px;
          border-radius: 6px; font-size: 14px; cursor: pointer; transition: background 0.2s; font-weight: 500; display: flex; align-items: center;">
        <svg xmlns="http://www.w3.org/2000/svg" width="16" height="16" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" style="margin-right: 6px;">
          <path d="M21 15v4a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2v-4"></path>
          <polyline points="7 10 12 15 17 10"></polyline>
          <line x1="12" y1="15" x2="12" y2="3"></line>
        </svg>
        Download
      </button>
    </div>
    <div id="result_content" style="white-space: pre-wrap; font-size: 15px; line-height: 1.6; color: #334155;"></div>
  </div>

  <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');

    @keyframes spin {
      0% { transform: rotate(0deg); }
      100% { transform: rotate(360deg); }
    }

    button:hover {
      background: #1d4ed8;
    }

    button[onclick="downloadResults"]:hover {
      background: #e2e8f0;
    }

    select, input, textarea {
      font-family: 'Inter', sans-serif;
      outline: none;
      transition: border-color 0.2s;
    }

    select:focus, input:focus, textarea:focus {
      border-color: #93c5fd !important;
      box-shadow: 0 0 0 3px rgba(59, 130, 246, 0.1);
    }

    #drop_zone {
      transition: all 0.2s;
    }

    #drop_zone.active {
      border-color: #3b82f6;
      background-color: rgba(59, 130, 246, 0.05);
    }

    #result_content h3 {
      color: #2563eb;
      margin-top: 24px;
      margin-bottom: 12px;
      font-size: 17px;
      font-weight: 600;
    }

    #result_content ul {
      margin-top: 12px;
      padding-left: 20px;
    }

    #result_content li {
      margin-bottom: 8px;
    }

    @media screen and (max-width: 768px) {
      .flex-container {
        flex-direction: column;
      }
    }
  </style>

  <script>
    // Show/hide custom query field based on analysis type selection
    document.getElementById('analysis_type').addEventListener('change', function() {
      const customQueryDiv = document.getElementById('custom_query_div');
      if (this.value === 'custom') {
        customQueryDiv.style.display = 'block';
      } else {
        customQueryDiv.style.display = 'none';
      }
    });

    // File upload handling
    const dropZone = document.getElementById('drop_zone');
    const fileInput = document.getElementById('fileInput');
    const documentText = document.getElementById('documentText');
    const fileName = document.getElementById('file_name');

    // File input change handler
    fileInput.addEventListener('change', handleFileSelect, false);

    // Drag and drop handlers
    dropZone.addEventListener('dragover', function(e) {
      e.stopPropagation();
      e.preventDefault();
      e.dataTransfer.dropEffect = 'copy';
      this.classList.add('active');
    });

    dropZone.addEventListener('dragleave', function() {
      this.classList.remove('active');
    });

    dropZone.addEventListener('drop', function(e) {
      e.stopPropagation();
      e.preventDefault();
      this.classList.remove('active');

      const files = e.dataTransfer.files;
      if (files.length > 0) {
        handleFile(files[0]);
      }
    });

    function handleFileSelect(e) {
      const files = e.target.files;
      if (files.length > 0) {
        handleFile(files[0]);
      }
    }

    function handleFile(file) {
      if (file.type !== 'text/plain') {
        alert('Please upload a text file (.txt)');
        return;
      }

      fileName.textContent = file.name;

      const reader = new FileReader();
      reader.onload = function(e) {
        documentText.value = e.target.result;
      };
      reader.readAsText(file);
    }

    function analyzeDocument() {
      const documentTextContent = document.getElementById('documentText').value;
      const documentLang = document.getElementById('document_lang').value;
      const analysisType = document.getElementById('analysis_type').value;
      const responseLang = document.getElementById('response_lang').value;
      let customQuery = '';

      if (analysisType === 'custom') {
        customQuery = document.getElementById('custom_query').value;
        if (!customQuery) {
          alert('Please enter a custom query');
          return;
        }
      }

      if (!documentTextContent) {
        alert('Please enter document text or upload a file');
        return;
      }

      // Show loading indicator
      document.getElementById('loading').style.display = 'block';
      document.getElementById('results_container').style.display = 'none';

      // Prepare data to send to Python backend
      const data = {
        documentText: documentTextContent,
        documentLang: documentLang,
        analysisType: analysisType,
        customQuery: customQuery,
        responseLang: responseLang
      };

      // Send data to Python backend
      google.colab.kernel.invokeFunction('analyze_document', [JSON.stringify(data)], {});
    }

    function displayResults(results) {
      // Hide loading indicator
      document.getElementById('loading').style.display = 'none';

      // Show results container
      document.getElementById('results_container').style.display = 'block';

      // Display the formatted results
      document.getElementById('result_content').innerHTML = results;

      // Scroll to results
      document.getElementById('results_container').scrollIntoView({behavior: 'smooth'});
    }

    function downloadResults() {
      const resultContent = document.getElementById('result_content').innerText;
      const analysisType = document.getElementById('analysis_type').value;
      const blob = new Blob([resultContent], {type: 'text/plain'});
      const url = URL.createObjectURL(blob);

      const a = document.createElement('a');
      a.href = url;
      a.download = `legal_analysis_${analysisType}_${new Date().toISOString().slice(0,10)}.txt`;
      document.body.appendChild(a);
      a.click();
      document.body.removeChild(a);
      URL.revokeObjectURL(url);
    }
  </script>
</div>
"""

# Display the frontend
display(HTML(frontend_html))

# Function to process different analysis types
def generate_query_based_on_analysis(document_text, analysis_type, custom_query, document_lang):
    """Generate appropriate query based on analysis type"""
    if analysis_type == "custom":
        return custom_query

    analysis_queries = {
        "summary": f"Summarize the following legal document in clear, concise language. Identify the main purpose, key provisions, and important points: {document_text}",
        "legal_terms": f"Extract and explain all important legal terms and jargon from this document: {document_text}",
        "obligations": f"Identify all legal obligations, requirements, and responsibilities mentioned in this document: {document_text}",
        "dates": f"Extract all dates, deadlines, and time-sensitive information from this document and explain their significance: {document_text}",
        "parties": f"Identify all parties mentioned in this document and describe their roles and relationships: {document_text}"
    }

    return analysis_queries.get(analysis_type, "Summarize this document.")

# Register callback function to process document analysis
def analyze_document_callback(data_json):
    # Parse the input data
    data = json.loads(data_json)
    document_text = data["documentText"]
    document_lang = data["documentLang"]
    analysis_type = data["analysisType"]
    custom_query = data["customQuery"]
    response_lang = data["responseLang"]

    # Generate the appropriate query
    query = generate_query_based_on_analysis(document_text, analysis_type, custom_query, document_lang)

    # Add language instruction
    if response_lang != "english":
        query += f" Reply in {response_lang}."

    try:
        # Create a document for the query system to use
        from langchain.docstore.document import Document
        from langchain_text_splitters import RecursiveCharacterTextSplitter

        # Split the document into chunks
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
        chunks = text_splitter.split_text(document_text)
        temp_docs = [Document(page_content=chunk) for chunk in chunks]

        # Create a temporary vector store with these documents
        temp_docsearch = PineconeVectorStore.from_documents(
            documents=temp_docs,
            index_name=index_name,
            embedding=embeddings,
            namespace="temp_analysis_" + str(int(time.time()))  # Create a temporary namespace
        )

        # Create a temporary QA system
        temp_qa = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type="stuff",
            retriever=temp_docsearch.as_retriever()
        )

        # Get results
        result = temp_qa.invoke(query)

        # Format the results based on analysis type
        answer = format_results(result["result"], analysis_type)

        # Return the result to the frontend
        output.eval_js(f'displayResults({json.dumps(answer)})')

    except Exception as e:
        error_message = f"<h3 style='color: #ef4444; margin-top: 0;'>Error</h3><p>{str(e)}</p>"
        output.eval_js(f'displayResults({json.dumps(error_message)})')

# Function to format results based on analysis type
def format_results(raw_result, analysis_type):
    """Format the results based on the analysis type"""
    formatted_result = raw_result

    # Add headers and formatting based on analysis type
    headers = {
        "summary": "<h3>Document Summary</h3>",
        "legal_terms": "<h3>Legal Terms Analysis</h3>",
        "obligations": "<h3>Legal Obligations</h3>",
        "dates": "<h3>Important Dates & Deadlines</h3>",
        "parties": "<h3>Parties Involved</h3>",
        "custom": "<h3>Analysis Results</h3>"
    }

    # Add the header for the specific analysis type
    return headers.get(analysis_type, "<h3>Analysis Results</h3>") + "<div>" + formatted_result + "</div>"

# Register the callback
output.register_callback('analyze_document', analyze_document_callback)